# 02 · Bronze to Silver

Tipagem, limpeza, deduplicação e os **campos canônicos** com os nomes definitivos.
Continua sendo uma linha por transação.

---

### Onde o trabalho acontece

A transformação é um `CREATE TABLE ... AS SELECT` que roda dentro do BigQuery:
1,3 milhão de linhas são lidas, convertidas e gravadas sem sair de lá.

O pandas aparece três vezes, todas com resultado pequeno: para espiar a Bronze,
para investigar a inconsistência de sete anos numa amostra, e para receber os
agregados de conferência.

### O que NÃO acontece aqui

Nenhuma feature de modelo. Distância, idade e período do dia nascem no `TRANSFORM`
do `CREATE MODEL`, no notebook 04 — o `TRANSFORM` é serializado junto com o modelo,
então em outubro a API do Vertex AI aplica exatamente aquele código.

In [ ]:
import pandas as pd
from google.cloud import bigquery

In [ ]:
!pip install --quiet --upgrade google-cloud-bigquery google-cloud-storage db-dtypes

In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Autenticado no Colab — use a conta dona do projeto")
except ImportError:
    print("Fora do Colab: usando as credenciais do ambiente")

## CONFIGURAÇÃO DOS PARÂMETROS:

In [ ]:
PROJECT_ID = "fraudflow-pdm-gps"
TABELA_BRONZE = f"{PROJECT_ID}.bronze.transactions"
TABELA_SILVER = f"{PROJECT_ID}.silver.transactions"

client = bigquery.Client(project=PROJECT_ID)

print(f"Origem  : {TABELA_BRONZE}")
print(f"Destino : {TABELA_SILVER}")

## Como a Bronze está hoje

`LIMIT 5` — descem cinco linhas, não 1,3 milhão.

In [ ]:
client.query(f"""
    SELECT trans_date_trans_time, unix_time, amt, merchant, dob, is_fraud
    FROM `{TABELA_BRONZE}`
    LIMIT 5
""").to_dataframe()

Duas coisas para reparar, e que a Silver resolve:

- `trans_date_trans_time` e `unix_time` discordam — a segunda parece anos mais antiga
- todo `merchant` começa com `fraud_`, inclusive em transação legítima

## A inconsistência de sete anos

Antes de decidir qual coluna de tempo usar, vale medir o problema. Trazemos uma
amostra de 50 mil linhas — **duas colunas apenas**, alguns megabytes — e o pandas
faz a conta.

Este é um uso legítimo do pandas: investigação sobre amostra, não transformação
sobre a base.

In [ ]:
amostra = client.query(f"""
    SELECT trans_date_trans_time, unix_time
    FROM `{TABELA_BRONZE}`
    LIMIT 50000
""").to_dataframe()

texto = pd.to_datetime(amostra['trans_date_trans_time'], format='%Y-%m-%d %H:%M:%S')
epoch = pd.to_datetime(pd.to_numeric(amostra['unix_time']), unit='s')
dif_dias = (texto - epoch).dt.total_seconds() / 86400

print(f"linhas analisadas : {len(amostra):,}")
print(f"deslocamento min  : {dif_dias.min():.4f} dias")
print(f"deslocamento max  : {dif_dias.max():.4f} dias")
print(f"valores distintos : {dif_dias.round(6).nunique()}")
print(f"equivale a        : {dif_dias.mean()/365.25:.4f} anos")

### O achado

**Exatamente 2557 dias — sete anos — constante.**

É material de slide, e a razão da regra do projeto: quem calculasse idade a partir
do `unix_time` deixaria todo mundo sete anos mais novo, sem nenhuma mensagem de
erro. Da Silver em diante, `trans_date_trans_time` é a única fonte de tempo.

## A transformação

Um `CREATE TABLE ... AS SELECT`. Leia com calma: é o coração da camada.

Repare no `QUALIFY ROW_NUMBER()` no final — é a deduplicação. Fazê-la aqui, e não
em pandas, garante que ela é global: o BigQuery enxerga a tabela inteira de uma vez.

In [ ]:
SQL_SILVER = f"""
-- ===========================================================================
-- Camada Silver — campos canônicos
--
-- Uma consulta só, executada dentro do BigQuery. 1,3 milhão de linhas são
-- lidas, transformadas e gravadas sem sair de lá: o notebook manda a ordem e
-- recebe a confirmação.
--
-- O que acontece:
--   1. tipagem com SAFE_CAST e SAFE.PARSE_* (o que não converte vira NULL)
--   2. resolução do conflito de sete anos entre as duas colunas de tempo
--   3. remoção do prefixo fraud_ dos comerciantes
--   4. descarte contável das linhas inválidas
--   5. deduplicação por trans_num
--
-- NENHUMA feature de modelo. Distância, idade e período do dia nascem no
-- TRANSFORM do CREATE MODEL, para viajarem junto com o modelo até o Vertex AI.
-- ===========================================================================

CREATE OR REPLACE TABLE `{PROJECT_ID}.silver.transactions`
PARTITION BY DATE(transaction_ts)
CLUSTER BY category
AS
WITH tipado AS (
  SELECT
    -- ---- identificação ----
    trans_num                                          AS transaction_id,
    cc_num                                             AS card_id,

    -- ---- tempo ----
    -- ARMADILHA: trans_date_trans_time diz 2019-2020 e unix_time, na MESMA
    -- linha, diz 2012-2013 — deslocamento constante de 2557 dias.
    -- trans_date_trans_time é a única fonte de tempo do projeto, e unix_time
    -- não é sequer lido aqui.
    SAFE.PARSE_TIMESTAMP('%Y-%m-%d %H:%M:%S',
                         trans_date_trans_time)        AS transaction_ts,

    -- ---- transação ----
    SAFE_CAST(amt AS FLOAT64)                          AS amount,
    category                                           AS category,
    -- os 693 comerciantes vêm com o prefixo fraud_, inclusive em transação
    -- legítima. É artefato do gerador, não vazamento — mas confunde e convida
    -- a filtro acidental.
    REGEXP_REPLACE(merchant, r'^fraud_', '')           AS merchant_name,

    -- ---- perfil do cliente (contexto estático do contrato do T2) ----
    SAFE.PARSE_DATE('%Y-%m-%d', dob)                   AS customer_dob,
    SAFE_CAST(lat AS FLOAT64)                          AS customer_lat,
    SAFE_CAST(`long` AS FLOAT64)                       AS customer_long,
    state                                              AS customer_state,
    SAFE_CAST(city_pop AS INT64)                       AS city_pop,

    -- ---- comerciante ----
    SAFE_CAST(merch_lat AS FLOAT64)                    AS merchant_lat,
    SAFE_CAST(merch_long AS FLOAT64)                   AS merchant_long,

    -- ---- rótulo e linhagem ----
    SAFE_CAST(is_fraud AS INT64)                       AS is_fraud,
    ingestion_timestamp,
    source_file

    -- Ausentes de propósito: unix_time (deslocado sete anos) e first, last,
    -- street, zip, gender e job — dados pessoais que o grupo decidiu não usar
    -- para decidir fraude. Não trazê-los torna a escolha verificável.
  FROM `{PROJECT_ID}.bronze.transactions`
)

SELECT
  transaction_id,
  card_id,
  transaction_ts,
  amount,
  category,
  merchant_name,
  customer_dob,
  customer_lat,
  customer_long,
  customer_state,
  city_pop,
  merchant_lat,
  merchant_long,
  is_fraud,
  ingestion_timestamp,
  source_file
FROM tipado
WHERE transaction_ts IS NOT NULL
  AND amount IS NOT NULL AND amount > 0
  AND customer_dob IS NOT NULL
  AND customer_dob < DATE(transaction_ts)
  AND customer_lat  BETWEEN  -90 AND  90
  AND customer_long BETWEEN -180 AND 180
  AND merchant_lat  BETWEEN  -90 AND  90
  AND merchant_long BETWEEN -180 AND 180
  AND is_fraud IN (0, 1)
-- Uma linha por transação. Em empate, fica a primeira ingerida.
QUALIFY ROW_NUMBER() OVER (
          PARTITION BY transaction_id
          ORDER BY ingestion_timestamp, source_file
        ) = 1
"""

In [ ]:
job = client.query(SQL_SILVER)
job.result()

print(f"Silver criada")
print(f"BigQuery leu       : {job.total_bytes_processed/1024**2:,.1f} MB")
print(f"Desceu para o Colab: 0 bytes — a transformacao rodou la")

## Conferência de qualidade

Uma consulta que varre a Silver inteira e devolve **uma linha**. O BigQuery faz o
trabalho; o pandas recebe o resumo.

In [ ]:
client.query(f"""
    SELECT
      COUNT(*)                                             AS linhas,
      COUNTIF(is_fraud = 1)                                AS fraudes,
      ROUND(100 * COUNTIF(is_fraud = 1) / COUNT(*), 3)     AS pct_fraude,
      COUNT(*) - COUNT(DISTINCT transaction_id)            AS duplicatas,
      COUNTIF(STARTS_WITH(merchant_name, 'fraud_'))        AS prefixo_restante,
      MIN(transaction_ts)                                  AS primeira,
      MAX(transaction_ts)                                  AS ultima
    FROM `{TABELA_SILVER}`
""").to_dataframe().T

Os três números que importam: **taxa de fraude perto de 0,58%**, **zero**
duplicatas e **zero** prefixo restante. E `ultima` tem que ficar antes de
01/07/2020 — se passar disso, o holdout vazou para a Bronze.

In [ ]:
chk = client.query(f'''
    SELECT
      COUNT(*)                                      AS linhas,
      COUNT(*) - COUNT(DISTINCT transaction_id)     AS duplicatas,
      COUNTIF(STARTS_WITH(merchant_name, 'fraud_')) AS prefixo,
      MAX(DATE(transaction_ts))                     AS ultima_data
    FROM `{TABELA_SILVER}`
''').to_dataframe().iloc[0]

# Se o formato do timestamp estiver errado, SAFE.PARSE_TIMESTAMP devolve NULL
# para tudo, o WHERE descarta todas as linhas e a Silver fica VAZIA. Sem este
# assert, os testes abaixo passariam numa tabela sem nenhuma linha.
assert chk['linhas'] > 1_200_000, (
    f"Silver com apenas {chk['linhas']:,} linhas. Provavel causa: o formato de "
    'trans_date_trans_time no CSV nao e %Y-%m-%d %H:%M:%S. Rode a celula de "
    'amostra e confira o formato real antes de seguir.'
)
assert chk['duplicatas'] == 0, 'ha transaction_id duplicado'
assert chk['prefixo'] == 0, 'ainda ha comerciante com o prefixo fraud_'
assert str(chk['ultima_data']) < '2020-07-01', 'o holdout vazou para a Bronze'
print('Qualidade ok')

## Conferência do contrato

Metadados da tabela — não custa query nenhuma.

In [ ]:
tabela = client.get_table(TABELA_SILVER)
colunas = [c.name for c in tabela.schema]

print(f"Linhas   : {tabela.num_rows:,}")
print(f"Particao : {tabela.time_partitioning.field}")
print(f"Cluster  : {tabela.clustering_fields}")
print()
print("  " + ", ".join(colunas))

canonicos = ['transaction_ts','customer_dob','customer_lat','customer_long',
             'merchant_lat','merchant_long','amount','category','city_pop','is_fraud']
faltando = [c for c in canonicos if c not in colunas]
assert not faltando, f'faltam campos canonicos: {faltando}'
assert not any('unix' in c.lower() for c in colunas), 'unix_time voltou para a Silver'
print('\nOs dez campos do contrato estao presentes, e unix_time nao')

## A fraude tem cara diferente?

Se as médias vierem iguais, alguma conversão saiu errada. O `GROUP BY` roda no
BigQuery e devolve duas linhas.

In [ ]:
client.query(f"""
    SELECT
      is_fraud,
      COUNT(*)                                          AS transacoes,
      ROUND(AVG(amount), 2)                             AS valor_medio,
      ROUND(AVG(city_pop))                              AS populacao_media,
      ROUND(AVG(EXTRACT(HOUR FROM transaction_ts)), 2)  AS hora_media
    FROM `{TABELA_SILVER}`
    GROUP BY is_fraud
    ORDER BY is_fraud
""").to_dataframe()

---

**Próximo:** `03_silver_to_gold.ipynb`